# Why YAPEAL, why wallet leakage

Issue #3. The argument for the first third of the presentation: one claim, one chart. Charts S1–S8 go to `figures/story/`, every chart is a function in `src/plots.py` so the dashboard (issue 12) can reuse it. The written argument and the expected counter-arguments are in `docs/story.md`.

S1–S5 reuse the chart functions from the leakage analysis (issue 6), S6 the ladder from issue 9.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd

from src import mappings as M
from src import plots, style
from src.data import load_customer_table, load_sow, load_sow_counterpart, monthly_active
from src.features import build_customer_features

style.apply()
pd.set_option("display.width", 200)
FOLDER = "story"

sow = load_sow()
cp = load_sow_counterpart()
feats = build_customer_features(load_customer_table())

monthly = sow.groupby("date").agg(spend=("total_amount", "sum")).join(monthly_active(sow))
quarterly_spend = sow.groupby("quarter")["total_amount"].sum()
fin = cp[cp["category"].isin(["cash", "savings"])].copy()

## S1 · a young, growing neobank

In [ ]:
ax = plots.active_customers(monthly, mark_peak=True)
plots.save_fig(ax.figure, "s1_active_customers", FOLDER)
a = monthly["active_customers"]
print(f"active customers: {a.iloc[0]:.0f} (Jan 21) → {a.max():.0f} (peak {a.idxmax():%b %y}) → {a.iloc[-1]:.0f} (Dec 23)")

The line ends below the peak: 1 648 in July 2023, 1 540 in December. On the slide: "689 → 1 540, peak 1 648". The jump in March 2021 is a single month and is not explained by the data.

## S2 · where the money goes

In [ ]:
cat_share = feats[[c for c in feats if c.startswith("cat_") and not c.endswith("_share")]].sum()
cat_share = (cat_share / cat_share.sum() * 100).rename(lambda c: c.replace("cat_", ""))
cat_share = cat_share[cat_share > 0.5]
ax = plots.category_shares(cat_share)
plots.save_fig(ax.figure, "s2_category_shares", FOLDER)
cat_share[["cash", "savings"]].round(1)

## S3 · 'cash' is not cash

In [ ]:
ax = plots.financial_counterparts(fin)
plots.save_fig(ax.figure, "s3_cash_is_not_cash", FOLDER)
named = fin[fin["top_counterpart"] != "other"]["total_amount"].sum() / fin["total_amount"].sum()
print(f"named counterparts: {named:.0%} of cash + savings, 'other': {1 - named:.0%}")

Every named counterpart is a financial or payment provider. `other` is about half of the two categories and stays unexplained, that is the first question in the Q&A.

## S4 · leakage grows

In [ ]:
fin["group"] = fin["top_counterpart"].map(M.LEAKAGE_GROUPS).fillna("other")
pct = fin.pivot_table(index="quarter", columns="group", values="total_amount", aggfunc="sum").fillna(0).div(quarterly_spend, axis=0) * 100
cols = ["revolut", "traditional banks", "crypto exchanges", "twint"]

ax = plots.share_lines(pct, cols, "card spend going to other financial providers, per quarter", "% of all card spend", ylim=5)
plots.save_fig(ax.figure, "s4_leakage_grows", FOLDER)
pct[cols].iloc[[0, -1]].round(1)

Bitcoin price and SNB policy rate belong next to this as separate small panels, not on a second axis. They come with the external data (issue 11).

## S5 · steady, not a one-off

In [ ]:
rev = fin[fin["top_counterpart"] == "revolut"].groupby("date")["n_customers"].sum()
rev_pct = rev / monthly["active_customers"] * 100
ax = plots.monthly_bar(rev_pct, "customers topping up Revolut, % of monthly active", "%")
plots.save_fig(ax.figure, "s5_one_in_twenty", FOLDER)
print(f"mean {rev_pct.mean():.1f} %, range {rev_pct.min():.1f}–{rev_pct.max():.1f} %, {len(rev_pct)} months")

## S6 · leakage and churn

Train set, customers active more than a year (n = 1 838). Leakage share = `cash + savings` as a share of spend (`leak_share`).

Why not `savings` alone, as in the ideation notebook: Revolut moved from `savings` to `cash` between December 2021 and May 2022 (both categories in parallel from January to April 2022, `savings` empty from May on). The customer file has no dates but the same labels, so `cat_savings` holds the Revolut transfers made before the switch. Churned customers are active earlier in the window, so `savings` alone partly measures *when* a customer was active. Among customers with leakage, 20 % of the churners' leakage sits in `savings`, 9 % of the stayers'. `cash + savings` does not depend on the switch.

In [ ]:
BINS = [-0.01, 0, 0.05, 0.2, 0.5, 1.0]
LABELS = ["0", "0–5 %", "5–20 %", "20–50 %", "> 50 %"]

old = feats[(feats["split"] == "train") & (feats["days_active"] > 365)].copy()
old["churn"] = old["churned"].astype(bool)


def ladder(col):
    g = old.groupby(pd.cut(old[col], BINS, labels=LABELS), observed=True)["churn"]
    return pd.DataFrame({"rate": g.mean(), "n": g.size()})


leak, savings = ladder("leak_share"), ladder("cat_savings_share")
ax = plots.churn_ladder(leak, "customers active > 1 year: churn by share of spend sent to other providers",
                        "share of card spend in 'cash' + 'savings'", reference=old["churn"].mean(), mute_first=True)
plots.save_fig(ax.figure, "s6_churn_ladder", FOLDER)

lk = old[old["leak_amount"] > 0]
print("share of leakage in 'savings', churned vs stayed:", (lk["cat_savings"] / lk["leak_amount"]).groupby(lk["churn"]).mean().round(2).to_dict())
pd.concat({"cash + savings": leak, "savings only": savings}, axis=1).round(2)

18 → 24 → 38 → 43 % among customers with leakage. The grey bar (no leakage at all, 34 %) mixes customers who never move money out with customers who barely used the card, which is why it sits above the 0–5 % bar. Wording for the slide: "customers who send a large part of their spend to other providers churn more, also at equal tenure", not "leakage causes churn". With controls the odds rise by about 10 % per +10 points of leakage share (issue 9).

## S6b · how we stress-tested our own finding

The first reading (`savings` only) against the corrected one (`cash + savings`), same customers, same scale. Among customers with leakage the ladder shrinks from 27 → 70 % to 18 → 43 %. The direction holds, the size does not.

In [ ]:
fig, _ = plots.churn_ladder_pair(
    savings, leak,
    titles=("first reading: 'savings' only", "corrected: 'cash' + 'savings'"),
    xlabels=("share of card spend in 'savings'", "share of card spend in 'cash' + 'savings'"),
    note="Revolut moved from 'savings' to 'cash' between Dec 2021 and May 2022. 'savings' alone therefore partly measures when a customer was active, "
         "and churned customers were active earlier.\nCustomers active > 1 year, n = 1 838. Grey bar right: no leakage at all.",
)
plots.save_fig(fig, "s6b_ladder_correction", FOLDER)

## S7 · we looked at the alternatives

In [ ]:
def group_share(category):
    d = cp[cp["category"] == category]
    p = d.pivot_table(index="quarter", columns="counterpart_group", values="total_amount", aggfunc="sum").fillna(0)
    return p.div(p.sum(axis=1), axis=0) * 100


groc, trans = group_share("groceries"), group_share("transport")
fig, _ = plots.share_panels([
    dict(pct=groc, columns=["coop", "migros", "discounter"], title="grocery wallet", ylabel="% of grocery spend", ylim=35),
    dict(pct=trans, columns=["fuel", "public transport", "ride hailing"], title="transport wallet", ylabel="% of transport spend", ylim=45),
])
plots.save_fig(fig, "s7_alternatives", FOLDER)
pd.concat({"groceries": groc.iloc[[0, -1]], "transport": trans.iloc[[0, -1]]}, axis=1).round(1)

Classic share of wallet works, but it is a 36-point time series per merchant without customers. It cannot be linked to churn or segments, see S8.

## S8 · the data constraint

In [ ]:
ax = plots.data_map(
    n_customers=len(feats),
    n_labelled=int((feats["split"] == "train").sum()),
    n_predict=int((feats["split"] == "predict").sum()),
    n_months=sow["date"].nunique(),
    n_counterparts=cp["top_counterpart"].nunique(),
)
plots.save_fig(ax.figure, "s8_data_map", FOLDER)